# 04 · Modeling — K-Means

**Owner:** Person 5 — Modeling (K-Means)  
**Run from:** the `notebooks/` folder

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `src` can be imported from notebooks/
warnings.filterwarnings("ignore")

import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
FIG = Path.cwd().parent / "reports" / "figures"


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from src.preprocess import load_clean_data, FEATURES, RANDOM_STATE

df = load_clean_data()
X = StandardScaler().fit_transform(df[FEATURES])

## PCA

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE).fit(X)
plt.plot(range(1, len(FEATURES) + 1), pca_full.explained_variance_ratio_.cumsum(), marker="o")
plt.xlabel("Components"); plt.ylabel("Cumulative explained variance"); plt.show()

pca = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X)
print("Variance kept by 3 components:", pca.explained_variance_ratio_.sum().round(3))

## Choosing k

In [ ]:
rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_pca)
    rows.append({"k": k, "inertia": km.inertia_, "silhouette": silhouette_score(X_pca, km.labels_), "davies_bouldin": davies_bouldin_score(X_pca, km.labels_)})
scores = pd.DataFrame(rows).set_index("k").round(3)
scores

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
scores["inertia"].plot(marker="o", ax=axes[0], title="Elbow method")
scores["silhouette"].plot(marker="o", ax=axes[1], title="Silhouette score")
plt.tight_layout(); plt.savefig(FIG / "choosing_k.png", dpi=150); plt.show()

**Decision:** silhouette peaks at k = 2, but two groups only split high vs low spenders. k = 4 gives actionable segments with an acceptable score → **k = 4**.

## Final model → saved to `app/model.pkl`

In [ ]:
from src.train import train

model, clustered = train(n_clusters=4)
print(f"Silhouette {model['silhouette']:.3f} · Davies-Bouldin {model['davies_bouldin']:.3f}")
model["profile"]

In [ ]:
import plotly.express as px
Xp = model["pca"].transform(model["scaler"].transform(clustered[FEATURES]))
fig = px.scatter_3d(x=Xp[:, 0], y=Xp[:, 1], z=Xp[:, 2], color=clustered["Cluster"].astype(str), opacity=0.6, labels={"color": "Cluster"})
fig.show()